In [1]:
import math

![alt text](second%20quantized.png "second quantized")
From https://doi.org/10.1063/5.0150291

Terms on the fifth line are irrelevant, they are coulomb interactions invoving the classically treated nuclei.

In [2]:
import numpy as np
from gbasis.parsers import parse_nwchem

print("def2-SVP Basis Set Loaded from NwChem Format:")
elec_basis_dict = parse_nwchem("6-31Geuw")
nuc_basis_dict = parse_nwchem("DZSNB.nw")

"""
for atom in basis_dict:
    print(f"Atom: {atom}")
    print(f"   Number of shells: {len(basis_dict[atom])}")
    for i, shell in enumerate(basis_dict[atom]):
        print(f"   Shell {i} has angular momentum {shell[0]}")
        print(f"   Shell {i} has exponents {shell[1]}")
        print(f"   Shell {i} has coefficients {shell[2].flatten()}")
"""

def2-SVP Basis Set Loaded from NwChem Format:


'\nfor atom in basis_dict:\n    print(f"Atom: {atom}")\n    print(f"   Number of shells: {len(basis_dict[atom])}")\n    for i, shell in enumerate(basis_dict[atom]):\n        print(f"   Shell {i} has angular momentum {shell[0]}")\n        print(f"   Shell {i} has exponents {shell[1]}")\n        print(f"   Shell {i} has coefficients {shell[2].flatten()}")\n'

In [3]:
import numpy as np
import scipy as sp
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

mass_proton = 1874.0 #mass of proton in atomic units (electron masses)

elec_atoms = ["H", "H"]
elec_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])

nuc_atoms = ["Q", "Q"]
nuc_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])

elec_basis = make_contractions(elec_basis_dict, elec_atoms, elec_atcoords, coord_types="cartesian")
nuc_basis = make_contractions(nuc_basis_dict, nuc_atoms, nuc_atcoords, coord_types="cartesian")

# symmetric orthogonalization of AOs. Szabo sec 3.4.5
elec_overlap = overlap_integral(elec_basis)
elec_ortho = np.linalg.inv(sp.linalg.sqrtm(elec_overlap)) # Transform needed to get orthogonal MOs

nuc_overlap = overlap_integral(nuc_basis)
nuc_ortho = np.linalg.inv(sp.linalg.sqrtm(nuc_overlap)) # Transform needed to get orthogonal MOs

elec_ke = kinetic_energy_integral(elec_basis, transform=elec_ortho)
print(elec_ke)

nuc_ke = kinetic_energy_integral(nuc_basis, transform=nuc_ortho) / mass_proton
print(nuc_ke)

elec_elec_coulomb = electron_repulsion_integral(elec_basis, transform=elec_ortho)

#test = overlap_integral(nuc_basis, transform=nuc_ortho)
#test2 = overlap_integral(elec_basis, transform=elec_ortho)
#print(test2)

[[ 2.04051166 -0.52903325 -0.39647675  0.11670815]
 [-0.52903325  0.46068821  0.11670815 -0.12331951]
 [-0.39647675  0.11670815  2.04051166 -0.52903325]
 [ 0.11670815 -0.12331951 -0.52903325  0.46068821]]
[[ 4.86721300e-02 -1.29717130e-02 -5.08875347e-09  5.85991222e-09]
 [-1.29717130e-02  1.62033761e-02  5.85991206e-09 -6.91102362e-09]
 [-5.08875347e-09  5.85991206e-09  4.86721300e-02 -1.29717130e-02]
 [ 5.85991222e-09 -6.91102362e-09 -1.29717130e-02  1.62033761e-02]]


In [4]:
# Construct representation of CAR (fermionic creation/annihilation operators)
ident = [[1, 0], [0, 1]]
pauli_x = [[0, 1], [1, 0]]
pauli_y = [[0, -1j], [1j, 0]]
pauli_z = [[1, 0], [0, -1]]

modes = 2

# create representation of system of N distinguishable spins
pauli_xs = []
pauli_ys = []
pauli_zs = []

spin_raise = []
spin_lower = []

create = []
annihilate = []

for i in range(modes):
    x = y = z = [1]
    for j in range(modes):
        if i == j:
            x = np.kron(x, pauli_x)
            y = np.kron(y, pauli_y)
            z = np.kron(z, pauli_z)
        else:
            x = np.kron(x, ident)
            y = np.kron(y, ident)
            z = np.kron(z, ident)

    pauli_xs.append(x)
    pauli_ys.append(y)
    pauli_zs.append(z)

    spin_raise.append(0.5*(x+1j*y))
    spin_lower.append(0.5*(x-1j*y))

# construct fermionic creation/annihilation operators via JWT
for i in range(modes):
    sum = np.zeros([2**modes, 2**modes])
    for j in range(i):
        sum = sum + (spin_raise[j] @ spin_lower[j])

    create_op = sp.linalg.expm(1j*math.pi*sum) @ spin_raise[i]
    annihilate_op = sp.linalg.expm(-1j*math.pi*sum) @ spin_lower[i]

    annihilate.append(annihilate_op)
    create.append(create_op)

for i in range(modes):
    for j in range(modes):
        #commutator = annihilate[i] @ annihilate[j] + annihilate[j] @ annihilate[i]
        commutator = create[i] @ create[j] + create[j] @ create[i]
        #commutator = annihilate[i] @ create[j] + create[j] @ annihilate[i]
        #print(commutator)

        if(np.allclose(commutator, np.eye(2**modes))):
            print(1)
        elif(np.allclose(commutator, np.zeros([2**modes, 2**modes]))):
            print(0)
        else:
            print("?")

print("bruh")


0
0
0
0
bruh


In [2]:
import math
import numpy as np
from gbasis.parsers import parse_nwchem

elec_basis_dict = parse_nwchem("6-31G.nw")
nuc_basis_dict = parse_nwchem("DZSNB.nw")

"""
for atom in basis_dict:
    print(f"Atom: {atom}")
    print(f"   Number of shells: {len(basis_dict[atom])}")
    for i, shell in enumerate(basis_dict[atom]):
        print(f"   Shell {i} has angular momentum {shell[0]}")
        print(f"   Shell {i} has exponents {shell[1]}")
        print(f"   Shell {i} has coefficients {shell[2].flatten()}")
"""
import numpy as np
import scipy as sp
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

mass_proton = 1874.0  #mass of proton in atomic units (electron masses)

# Centers of electron orbitals (just the H atoms position)
elec_atoms = ["H", "H"]
elec_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])

# Centers of nuclear orbitals for protons
nuc_atoms = ["Q", "Q"]
nuc_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])


# Construct full molecular orbital  basis from atomic orbitals centered at the positions "atcoords"
elec_basis = make_contractions(elec_basis_dict, elec_atoms, elec_atcoords, coord_types="cartesian")
nuc_basis = make_contractions(nuc_basis_dict, nuc_atoms, nuc_atcoords, coord_types="cartesian")
full_basis = elec_basis + nuc_basis # Bases are tuples of contracted gaussian objects. NOTE that the overlap

In [3]:
# symmetric orthogonalization of AOs. Szabo sec 3.4.5
elec_overlap = overlap_integral(elec_basis)
elec_ortho = np.linalg.inv(sp.linalg.sqrtm(elec_overlap))  # Transform needed to get orthogonal MOs

nuc_overlap = overlap_integral(nuc_basis)
nuc_ortho = np.linalg.inv(sp.linalg.sqrtm(nuc_overlap))  # Transform needed to get orthogonal MOs

full_ortho = np.block([[elec_ortho, np.zeros((len(elec_basis), len(nuc_basis)))],
                       [np.zeros((len(nuc_basis), len(elec_basis))), nuc_ortho]])

full_overlap = overlap_integral(full_basis, transform=full_ortho)
print(np.matrix.round(full_overlap, 2))
# Note the "non-zero" overlap between nuclear and electronic orbitals. They actually are orthogonal in the Hilbert space though.


[[ 1.    0.   -0.   -0.    0.01  0.47 -0.01 -0.02]
 [ 0.    1.   -0.    0.   -0.02 -0.07 -0.    0.04]
 [-0.    0.    1.   -0.   -0.01 -0.02  0.01  0.47]
 [-0.    0.   -0.    1.   -0.    0.04 -0.02 -0.07]
 [ 0.01 -0.02 -0.01 -0.    1.   -0.    0.    0.  ]
 [ 0.47 -0.07 -0.02  0.04 -0.    1.   -0.   -0.  ]
 [-0.01 -0.    0.01 -0.02  0.   -0.    1.    0.  ]
 [-0.02  0.04  0.47 -0.07  0.   -0.    0.    1.  ]]


In [23]:
c = 1/math.sqrt(2)
h = [[c, c],
     [c, -c]]
symmetrize = np.kron(h,np.eye(2))
print(symmetrize)

sym_overlap = overlap_integral(elec_basis, transform=symmetrize) # Transformation to symmetric/antisymmetric orbitals. No longer unit length
sym_ortho = np.linalg.inv(sp.linalg.sqrtm(sym_overlap)) @ symmetrize

print(np.matrix.round(overlap_integral(elec_basis, transform=sym_ortho), 21))
print(np.matrix.round(kinetic_energy_integral(elec_basis, transform=sym_ortho), 2))



#print(np.matrix.round(sym_overlap, 2))
#print(np.matrix.round(sym_ortho, 2))

[[ 0.70710678  0.          0.70710678  0.        ]
 [ 0.          0.70710678  0.          0.70710678]
 [ 0.70710678  0.         -0.70710678 -0.        ]
 [ 0.          0.70710678 -0.         -0.70710678]]
[[ 1.  0.  0.  0.]
 [ 0.  1. -0.  0.]
 [ 0. -0.  1.  0.]
 [ 0. -0.  0.  1.]]
[[ 1.64 -0.41 -0.    0.  ]
 [-0.41  0.34 -0.    0.  ]
 [ 0.   -0.    2.44 -0.65]
 [ 0.   -0.   -0.65  0.58]]


In [4]:
elec_ke = kinetic_energy_integral(elec_basis, transform=elec_ortho)
print(elec_ke)

nuc_ke = kinetic_energy_integral(nuc_basis, transform=nuc_ortho) / mass_proton
print(nuc_ke)

coulomb = electron_repulsion_integral(full_basis, transform=full_ortho)



[[ 2.04051166 -0.52903325 -0.39647675  0.11670815]
 [-0.52903325  0.46068821  0.11670815 -0.12331951]
 [-0.39647675  0.11670815  2.04051166 -0.52903325]
 [ 0.11670815 -0.12331951 -0.52903325  0.46068821]]
[[ 4.86721300e-02 -1.29717130e-02 -5.08875347e-09  5.85991222e-09]
 [-1.29717130e-02  1.62033761e-02  5.85991206e-09 -6.91102362e-09]
 [-5.08875347e-09  5.85991206e-09  4.86721300e-02 -1.29717130e-02]
 [ 5.85991222e-09 -6.91102362e-09 -1.29717130e-02  1.62033761e-02]]


In [5]:
print(coulomb[0,0,0,0])

1.191465046216982


In [ ]:

#test = overlap_integral(nuc_basis, transform=nuc_ortho)
#test2 = overlap_integral(elec_basis, transform=elec_ortho)
#print(test2)
# Construct representation of CAR (fermionic creation/annihilation operators)
ident = [[1, 0], [0, 1]]
pauli_x = [[0, 1], [1, 0]]
pauli_y = [[0, -1j], [1j, 0]]
pauli_z = [[1, 0], [0, -1]]

modes = 2

# create representation of system of N distinguishable spins
pauli_xs = []
pauli_ys = []
pauli_zs = []

spin_raise = []
spin_lower = []

create = []
annihilate = []

for i in range(modes):
    x = y = z = [1]
    for j in range(modes):
        if i == j:
            x = np.kron(x, pauli_x)
            y = np.kron(y, pauli_y)
            z = np.kron(z, pauli_z)
        else:
            x = np.kron(x, ident)
            y = np.kron(y, ident)
            z = np.kron(z, ident)

    pauli_xs.append(x)
    pauli_ys.append(y)
    pauli_zs.append(z)

    spin_raise.append(0.5 * (x + 1j * y))
    spin_lower.append(0.5 * (x - 1j * y))

# construct fermionic creation/annihilation operators via JWT
for i in range(modes):
    sum = np.zeros([2 ** modes, 2 ** modes])
    for j in range(i):
        sum = sum + (spin_raise[j] @ spin_lower[j])

    create_op = sp.linalg.expm(1j * math.pi * sum) @ spin_raise[i]
    annihilate_op = sp.linalg.expm(-1j * math.pi * sum) @ spin_lower[i]

    annihilate.append(annihilate_op)
    create.append(create_op)

for i in range(modes):
    for j in range(modes):
        #commutator = annihilate[i] @ annihilate[j] + annihilate[j] @ annihilate[i]
        commutator = create[i] @ create[j] + create[j] @ create[i]
        #commutator = annihilate[i] @ create[j] + create[j] @ annihilate[i]
        #print(commutator)

        if (np.allclose(commutator, np.eye(2 ** modes))):
            print(1)
        elif (np.allclose(commutator, np.zeros([2 ** modes, 2 ** modes]))):
            print(0)
        else:
            print("?")

print("bruh")



In [6]:
print(type(elec_basis))z

<class 'tuple'>
